In [20]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START, END
from typing import Literal,TypedDict,Annotated, List
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

True

In [21]:
model = ChatGoogleGenerativeAI(model="gemini-flash-latest")

In [17]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
   messages : Annotated[List[BaseMessage], add_messages]

In [22]:
def chat_node(state: ChatState) -> ChatState:
    #   take the quary from the use 
    messages = state["messages"]

    # send that quary to llm 
    response = model.invoke(messages)

    # response store in the state
    return {"messages": [response]}

In [23]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

In [24]:
initial_state = {"messages": [HumanMessage(content="What is the capital of France?")]}

In [ ]:
thread_id = '1'  # You can use any identifier for the conversation thread

while True:
    user_input = input("You: ")
    print(f"User input: {user_input}")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting chat.")
        break

    initial_state = {"messages": [HumanMessage(content=user_input)]}
    response = chatbot.invoke(initial_state, config={"configurable": {"thread_id": thread_id}})
    print(f"Bot: {response['messages'][0].content}")

User input: hii my name is imtiyaz A. khan
Bot: hii my name is imtiyaz A. khan
User input: what is my name 
Bot: hii my name is imtiyaz A. khan
User input: how do you know that my name is imtiyaz
Bot: hii my name is imtiyaz A. khan
